# Results for model: yentinglin/llama-3-taiwan-70b-instruct

In [ ]:
import polars as pl
import numpy as np
import xgboost as xgb
from math import ceil

data = pl.read_parquet('./jane_street_data/train.parquet')

TOP_QUANTILE = 0.15
TOP_QUANTILE_SIZE = int(ceil(data.height * TOP_QUANTILE))

def global_top_quantile(df):
    return df.sort('responder_6', descending=True).head(TOP_QUANTILE_SIZE)

group_by_date = data.groupby('date_id').then(global_top_quantile)
result = group_by_date.with_column(
    pl.concat_list([
        pl.lit(col).cast(pl.Float32)
        for col in data.columns
        if 'feature' in col
    ])
).with_column(pl.lit(__yesno__="True").cast(pl.Boolean))
result

x_train = result.select(pl.all().exclude('responder_6')).to_dask().compute().values
y_train = result['responder_6'].to_dask().compute().values

model = xgb.XGBRegressor(max_depth=2, n_estimators=10).fit(x_train, y_train)
model